In [11]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import numpy as np


In [12]:
from manify.curvature_estimation.delta_hyperbolicity import delta_hyperbolicity

In [ ]:
class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        B = x.shape[0]
        return x.view(B, -1)

def get_delta(loader, device, batch_size=500):
    """
    computes delta value for image data by extracting features using VGG network;
    input -- data loader for images
    """
    vgg = torchvision.models.vgg16(pretrained=True)
    vgg_feats = vgg.features
    vgg_classifier = nn.Sequential(*list(vgg.classifier.children())[:-1])

    vgg_part = nn.Sequential(vgg_feats, Flatten(), vgg_classifier).to(device)
    vgg_part.eval()

    all_features = []
    for i, (batch, _) in enumerate(loader):
        if i % 10 == 0:
            print(f'{i} Iterations done')
        resize = transforms.Resize((224, 224))
        batch = torch.stack([resize(img) for img in batch])
        with torch.no_grad():
            batch = batch.to(device)
            all_features.append(vgg_part(batch))

    all_features = torch.cat(all_features)
    idx = torch.randperm(len(all_features), device=device)[:batch_size]
    all_features_small = all_features[idx]

    dists = torch.cdist(all_features_small, all_features_small, p=2)
    delta = delta_hyperbolicity(dists)
    
    diam = torch.max(dists)
    return delta, diam

In [14]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Define transformations for test data
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Create training dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='../data', 
    train=True,
    download=True,
    transform=transform_train
)

# Create test dataset
test_dataset = torchvision.datasets.CIFAR10(
    root='../data', 
    train=False,
    download=True,
    transform=transform_test
)

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=128,
    shuffle=True, 
    num_workers=2
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    batch_size=128,
    shuffle=False, 
    num_workers=2
)

In [17]:
delta, diam = get_delta(test_loader,device='cuda')
print(f'Relative Delta: {2 * delta / diam}')

/home/dylan/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dylan/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


0 Iterations done
10 Iterations done
20 Iterations done
30 Iterations done
40 Iterations done
50 Iterations done
60 Iterations done
70 Iterations done
Relative Delta: 0.22346840798854828


In [16]:
delta_hyperbolicity

<function manify.curvature_estimation.delta_hyperbolicity.delta_hyperbolicity(dists: torch.Tensor)>